In [12]:
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough
from atlassian import Jira

# model="llama3.1:8b-instruct-q8_0"
model="llama3.1"
llm = ChatOllama(model=model, temperature=0)

In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
try:
    jira = Jira(
    url=os.getenv("url"),
    username=os.getenv("username"),
    password=os.getenv("password"),
    cloud=True)
except Exception as e:
    print(f"Unable to login to jira, Error: {e}")

In [3]:
import pandas as pd

def createJiraTaskFromLocalCSVFile(csvPath:str):
    df = pd.read_csv(csvPath)
    for index, row in df.iterrows():
        if pd.notna(row['summary']) & pd.isna(row['jira']):
            fields = {'project':{'key':'AN30'},'issuetype': {'name': 'Task'},'summary': row['summary'], 'description':row['description'], 'assignee':{'id':row['assignee']}}

            # res will contain {'id': '2784859', 'key': 'AN30-6067', 'self': 'link to the json response'}
            res=jira.issue_create(fields)

            df.at[index, 'jira'] = f'https://amagiengg.atlassian.net/browse/{res['key']}'
    df.to_csv(csvPath, index=False)
    
# createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv') 
# key=2784795

def getIssue():
    print(jira.issue(key))

# getIssue()


In [13]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

store = {}

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a highly capable AI assistant tasked with understanding user queries and responding with the appropriate function from the list provided below. Your response must adhere to the following constraints:

            Functions:
                - create_jira_task_from_endpoint(endpoint)

            Constraints:
                - Only use the functions listed above. Do not generate or suggest any other functions.
                - Ensure that the function you generate directly addresses the query made by the user.
                - If the query does not correspond to any function you are allowed to use, respond with an empty string ''.
                - Include only the function name and arguments in your response, without any additional text.
                - If there is no path mentioned in the query then respond with empty string ''.
                - If the question is related to any task we did in the current session then you should give relevant answer.
                - When the query is asking to create jira tickets then it should also have a https URL otherwise respond with empty string ''.
                - While responding replace [URL] with the actual URL provided by the user

            Examples:
                - Query: "Can you read the csv from [URL] and create jira for each items" 
                Response: "create_jira_task_from_endpoint([URL])"
                - Query: "read the csv in the path [URL] and create jira for each items" 
                Response: "create_jira_task_from_endpoint([URL])"
                - Query: "csv read jira [URL]"
                Response: ''
                - Query: "csv read [URL]"
                Response: ''
                - Query: "jira tickets [URL]"
                Response: ''

            Be mindful that only the functions defined above are valid, and the response must match the function signature exactly.
            """
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain =  RunnablePassthrough.assign(messages=itemgetter("messages")) | prompt | llm 

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

config = {"configurable": {"session_id": "new"}}


In [1]:
import re
def process_google_sheet_endpoint(endpoint: str):
    pattern = r"https://docs.google.com/spreadsheets/d/([a-zA-Z0-9-_]+).*?gid=([0-9]+)"
    match = re.search(pattern, endpoint)

    if match:
        sheet_id = match.group(1)
        gid = match.group(2)
        print(f"Sheet ID: {sheet_id}")
        print(f"GID: {gid}")
    else:
        print("URL format is not recognized.")
        
process_google_sheet_endpoint('https://docs.google.com/spreadsheets/d/1mPO6-Ta-3t3ECBXBvROxzebG9DwiDe7Ran5quHeuVa0/edit?gid=0#gid=0')

Sheet ID: 1mPO6-Ta-3t3ECBXBvROxzebG9DwiDe7Ran5quHeuVa0
GID: 0


In [37]:
import requests 
from io import StringIO

def create_jira_task_from_endpoint(endpoint:str):
  response = requests.get(endpoint)
  csv_data = StringIO(response.content.decode('utf-8'))
  df = pd.read_csv(csv_data)
  print(df)
  for index, row in df.iterrows():
    try:
      print('try')
      if pd.notna(row['summary']) and pd.isna(row['jira']):
        fields = {'project':{'key':'AN30'},'issuetype': {'name': 'Task'},'summary': row['summary'], 'description':row['description'], 'assignee':{'id':row['assignee']}}
        print(fields)
      else:
          print('else')
    except:
      print('except')
      if pd.notna(row['summary']):
        fields = {'project':{'key':'AN30'},'issuetype': {'name': 'Task'},'summary': row['summary'], 'description':row['description'], 'assignee':{'id':row['assignee']}}
        print(fields)
      else:
          print('else')
       
    

create_jira_task_from_endpoint('https://docs.google.com/spreadsheets/d/1mPO6-Ta-3t3ECBXBvROxzebG9DwiDe7Ran5quHeuVa0/export?format=csv')

                                       summary  \
0  Handle multiple sub folders inside dropzone   
1  Handle multiple sub folders inside dropzone   

                                   description assignee                 jira  
0  Handle multiple sub folders inside dropzone    varun  https://example.com  
1  Handle multiple sub folders inside dropzone    manoj                  NaN  
try
else
try
{'project': {'key': 'AN30'}, 'issuetype': {'name': 'Task'}, 'summary': 'Handle multiple sub folders inside dropzone', 'description': 'Handle multiple sub folders inside dropzone', 'assignee': {'id': 'manoj'}}


In [16]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="read the csv in the path https://docs.google.com/spreadsheets/d/1asdfasdfasdfasdf/export?format=csv and create jira for each items")]},
    config=config,
)
res = response.content.strip()
print(res)



# for r in with_message_history.invoke(
#     {"messages":[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list.csv'")]},
#     config=config,
# ):
#     print(r.content, end="", flush=True)

create_jira_task_from_endpoint(https://docs.google.com/spreadsheets/d/1asdfasdfasdfasdf/export?format=csv)


In [11]:
print(store)

{'new': InMemoryChatMessageHistory(messages=[HumanMessage(content='create jira tickets using the csv in the path https://docs.google.com/spreadsheets/d/1mPO6-Ta-3t3ECBXBvROxzeasdfasdfbG9DwiDe7Ran5quHeuVa0/export?format=csv'), AIMessage(content='create_jira_task_from_endpoint(https://docs.google.com/spreadsheets/d/1mPO6-Ta-3t3ECBXBvROxzeasdfasdfbG9DwiDe7Ran5quHeuVa0/export?format=csv)', response_metadata={'model': 'llama3.1:8b-instruct-q8_0', 'created_at': '2024-08-25T12:40:41.604098Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 8821743375, 'load_duration': 29409625, 'prompt_eval_count': 428, 'prompt_eval_duration': 5889420000, 'eval_count': 57, 'eval_duration': 2892778000}, id='run-d61d01aa-d73b-44de-9b9b-8e276ff4f1cb-0'), HumanMessage(content='create jira tickets using the csv in the path https://docs.google.com/spreadsheets/d/1mPO6-Ta-123123/export?format=csv'), AIMessage(content="''", response_metadata={'model': 'llama3.1:

In [155]:
from langchain_core.messages import SystemMessage

def updateSessionHistory(session_id: str):
    if session_id in store:
        get_session_history(session_id).add_message(SystemMessage("""We have created the following jira \n- Have popover with the same width as the field width: \n- Build a dynamic deployment action for particular feature branch from PR itself: """))

updateSessionHistory('new')

In [156]:
print(store)

{'new': InMemoryChatMessageHistory(messages=[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list.csv'"), AIMessage(content="createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list.csv')", response_metadata={'model': 'llama3.1', 'created_at': '2024-08-24T17:32:49.808062Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 3747059125, 'load_duration': 28551083, 'prompt_eval_count': 398, 'prompt_eval_duration': 3181510000, 'eval_count': 18, 'eval_duration': 532159000}, id='run-fdafdab9-ca4a-4cb5-b13d-e731ae1e62bf-0'), SystemMessage(content='We have created the following jira \n- Have popover with the same width as the field width: https://amagiengg.atlassian.net/browse/AN30-6067\n- Build a dynamic deployment action for particular feature branch from PR itself: https://amagiengg.atlassian.net/browse/AN30-6068')])}


In [ ]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="which all jira have we created")]},
    config=config,
)
res = response.content.strip()
print(res)

In [163]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="how many jira did we create")]},
    config=config,
)
res = response.content.strip()
print(res)

3


In [ ]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="can you please share those?")]},
    config=config,
)
res = response.content.strip()
print(res)

In [160]:
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="create jira tickets using the csv in the path  '../csvFiles/jira_task_list1.csv'")]},
    config=config,
)
res = response.content.strip()
print(res)


createJiraTaskFromLocalCSVFile('../csvFiles/jira_task_list1.csv')


In [161]:
def updateSessionHistory(session_id: str):
    if session_id in store:
        get_session_history(session_id).add_message(SystemMessage("""We have created the following jira \n- Have popover with the same width as the field width: \n- Build a dynamic deployment action for particular feature branch from PR itself: \n- Show delivery status inside delivery details: """))

updateSessionHistory('new')

In [132]:

if(len(res)>2):
    try:
        eval(res)
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Unexpected response:", res)

Unexpected response: ''


In [29]:
ar=[1,2,3]
print(len(ar))

3
